# CNN Primitives in PyTorch

In [ ]:
import torch
import torch.nn as nn

def n_params(layer):
    """Total number of trainable parameters in a layer."""
    return sum(p.numel() for p in layer.parameters())

## 1. The Conv2d Layer

A convolutional layer slides a small **kernel** (filter) across the spatial dimensions.
Tensors always have shape `(batch, channels, height, width)`.

Let us create one layer and pass a batch of images through it.

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)

x = torch.randn(8, 3, 32, 32)   # 8 RGB images, 32x32
y = conv(x)

print(f'Input  shape: {tuple(x.shape)}   (batch, C_in, H, W)')
print(f'Output shape: {tuple(y.shape)}  (batch, C_out, H_out, W_out)')
print()
print(f'Weight tensor: {tuple(conv.weight.shape)}  -> (C_out, C_in, K, K)')
print(f'Bias tensor:   {tuple(conv.bias.shape)}             -> (C_out,)')
print()
print(f'Total trainable parameters: {n_params(conv)}')

**Verify with formulas**

Output size (each spatial dimension, default stride=1, padding=0, dilation=1):
$$H_{out} = H_{in} - K + 1 = 32 - 3 + 1 = 30$$

Parameter count:
$$params = C_{out} \times C_{in} \times K \times K + C_{out} = 16 \times 3 \times 3 \times 3 + 16 = 448$$

Note: the parameter count **does not depend on the spatial size of the input** —
the same kernel is reused at every location (weight sharing).

In [ ]:
# Formula verification
h_out_formula = 32 - 3 + 1
params_formula = 16 * 3 * 3 * 3 + 16

print(f'Formula H_out = {h_out_formula},   PyTorch H_out = {y.shape[2]}  ->  match: {h_out_formula == y.shape[2]}')
print(f'Formula params = {params_formula}, PyTorch params = {n_params(conv)}  ->  match: {params_formula == n_params(conv)}')

## 2. Effect of Kernel Size

We create one Conv2d layer per kernel size, pass the same input, and read
the output shape and parameter count directly from PyTorch.

In [ ]:
x = torch.randn(1, 3, 32, 32)

print(f'{"Kernel":>8}  {"PyTorch output":>16}  {"PyTorch params":>16}  {"Formula H_out":>14}  {"Formula params":>15}')
print('-' * 78)
for k in [1, 3, 5, 7, 9]:
    conv = nn.Conv2d(3, 16, kernel_size=k)
    y = conv(x)
    h_formula = 32 - k + 1
    p_formula  = 16 * 3 * k * k + 16
    print(f'  {k}x{k}       {str(tuple(y.shape)):<18} {n_params(conv):>12,}    {h_formula}x{h_formula}           {p_formula:>10,}')

Larger kernels shrink the output and increase the parameter count **quadratically** with `K`:
$$params \propto K^2$$

## 3. Effect of Stride

Stride sets how many pixels the kernel shifts at each step. Stride > 1 downsamples the output.

In [ ]:
x = torch.randn(1, 3, 32, 32)

print(f'{"Stride":>8}  {"PyTorch output":>16}  {"PyTorch params":>16}  {"Formula H_out":>14}')
print('-' * 62)
for s in [1, 2, 4, 8]:
    conv = nn.Conv2d(3, 16, kernel_size=3, stride=s)
    y = conv(x)
    h_formula = (32 - 3) // s + 1
    print(f'     {s}      {str(tuple(y.shape)):<18} {n_params(conv):>12,}    {h_formula}x{h_formula}')

print()
print('Observation: all strides give the same parameter count.')
print('Stride controls spatial downsampling but never changes the number of weights.')

## 4. Effect of Padding

Padding adds zeros around the border of the input. Without it, every convolution shrinks
the spatial dimensions. A common goal is to preserve input size — this is called **same padding**.

For stride=1 and an odd kernel: `padding = kernel // 2` keeps `H_out == H_in`.

In [ ]:
x = torch.randn(1, 3, 32, 32)

print('-- Varying padding (kernel=3, stride=1) --')
print(f'{"Padding":>9}  {"PyTorch output":>16}  {"Formula H_out":>14}')
print('-' * 44)
for p in [0, 1, 2, 3]:
    conv = nn.Conv2d(3, 16, kernel_size=3, padding=p)
    y = conv(x)
    h_formula = 32 + 2*p - 3 + 1
    note = '  <- same as input' if y.shape[2] == 32 else ''
    print(f'    p={p}      {str(tuple(y.shape)):<18} {h_formula}x{h_formula}{note}')

print()
print('-- Same-padding rule applied to different kernels (padding = kernel // 2) --')
print(f'{"Kernel":>8}  {"Padding":>9}  {"PyTorch output":>16}')
print('-' * 38)
for k in [1, 3, 5, 7]:
    p = k // 2
    conv = nn.Conv2d(3, 16, kernel_size=k, padding=p)
    y = conv(x)
    print(f'  {k}x{k}        p={p}      {str(tuple(y.shape))}')

## 5. Effect of Dilation

Dilation inserts gaps between kernel elements. The kernel still has the same number of weights,
but it "sees" a larger region of the input — a larger **receptive field** — without extra parameters.

Effective receptive field: $RF = d \times (K - 1) + 1$

Setting `padding = dilation` keeps the output size equal to the input size (for kernel=3).

In [ ]:
x = torch.randn(1, 3, 32, 32)

print(f'{"Dilation":>10}  {"PyTorch output":>16}  {"PyTorch params":>16}  {"Receptive field":>18}')
print('-' * 68)
for d in [1, 2, 3, 4]:
    conv = nn.Conv2d(3, 16, kernel_size=3, dilation=d, padding=d)
    y = conv(x)
    rf = d * (3 - 1) + 1
    print(f'    d={d}        {str(tuple(y.shape)):<18} {n_params(conv):>12,}    {rf}x{rf}')

print()
print('Observation: parameter count is identical across all dilations.')
print('d=4 sees a 9x9 region of the input using the same 9 weights as d=1 (3x3 region).')

## 6. Parameter Savings vs Dense Layers

Why not use a `Linear` layer on the flattened image?
Let us build both options in PyTorch and compare their parameter counts directly.

In [ ]:
# Task: map a 3x32x32 image to 64 feature maps of the same spatial size (64x32x32)

# Option A: Conv2d with same padding
conv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1)

# Option B: Linear layer mapping flattened input to flattened output
in_features  = 3 * 32 * 32   # 3072
out_features = 64 * 32 * 32  # 65536
linear = nn.Linear(in_features, out_features)

# Run the same input through both
x = torch.randn(1, 3, 32, 32)

y_conv   = conv(x)
y_linear = linear(x.flatten(1))

print('Task: 3x32x32  ->  64x32x32')
print()
print(f'  Conv2d(3, 64, kernel=3, padding=1)')
print(f'    output shape : {tuple(y_conv.shape)}')
print(f'    parameters   : {n_params(conv):>12,}')
print()
print(f'  Linear({in_features}, {out_features})')
print(f'    output shape : {tuple(y_linear.shape)}  (spatial structure lost)')
print(f'    parameters   : {n_params(linear):>12,}')
print()
print(f'  Dense / Conv ratio: {n_params(linear) / n_params(conv):,.0f}x more parameters')
print()
print('Conv2d achieves this efficiency through:')
print('  1. Weight sharing    — the same kernel slides across all spatial positions')
print('  2. Local connectivity — each output depends only on a small neighbourhood')

## 7. Tracing Shapes Through VGG19

We load a pretrained VGG19 and use `torchinfo` to trace the actual tensor shape
and parameter count at every layer.

In [ ]:
import torchvision.models as models
from torchinfo import summary

vgg19 = models.vgg19()
summary(vgg19, input_size=(1, 3, 224, 224), col_names=["input_size", "output_size", "num_params"])